In [5]:
# ═════════════════════════════════════════════════════════════════
#  Airports Map - Global Coverage
#  Author: Germán Alberto Giménez Silva <ggerman@gmail.com>
#  Library: libgd-gis / ruby-libgd
#  Description: Major airports worldwide from OpenFlights.org.
#               Color by continent. Larger circles = major hubs.
# ═════════════════════════════════════════════════════════════════

require 'gd'
require 'gd/gis'
require 'net/http'
require 'csv'

def render(file_name)
    IRuby.display "<img src='#{file_name}' />", mime: "text/html"
end

def polaroid_frame(image, margin: 20, bottom_margin: 80, bg_color: [255, 255, 255, 255])
  new_width  = image.width + margin * 2
  new_height = image.height + margin + bottom_margin

  framed = GD::Image.new(new_width, new_height)
  framed.antialias = false
  bg = GD::Color.rgba(*bg_color)
  framed.filled_rectangle(0, 0, new_width, new_height, bg)
  framed.copy(image, margin, margin, 0, 0, image.width, image.height)
  framed
end

def continent_from_coords(lat, lon)
  if lat > 60; "arctic"
  elsif lat < -60; "antarctic"
  elsif lat > 23.5 && lon.between?(-125, -60); "north_america"
  elsif lat > 20 && lon.between?(-120, -30); "south_america"
  elsif lat > 35 && lon.between?(-10, 40); "europe"
  elsif lat > 10 && lon.between?(25, 65); "middle_east"
  elsif lat > 10 && lon.between?(65, 105); "asia_south"
  elsif lat > 25 && lon.between?(105, 145); "asia_east"
  elsif lat.between?(-40, 10) && lon.between?(110, 155); "oceania"
  elsif lat.between?(-35, 20) && lon.between?(-20, 55); "africa"
  else; "other"
  end
end

puts "Fetching airport data..."

url = "https://raw.githubusercontent.com/jpatokal/openflights/master/data/airports.dat"
response = Net::HTTP.get(URI(url))

airports = []
CSV.parse(response).each do |row|
  lat = row[6].to_f
  lon = row[7].to_f
  iata = row[4]
  name = row[1]

  next if iata.nil? || iata.empty? || iata.length != 3
  next unless lat.between?(-85, 85)
  next if name.to_s.downcase.include?("private") || name.to_s.downcase.include?("heliport")

  airports << { lon: lon, lat: lat, iata: iata, continent: continent_from_coords(lat, lon) }
end

puts "Retrieved #{airports.size} airports"

colors = {
  "north_america" => [50, 100, 200, 220],
  "south_america" => [100, 200, 100, 220],
  "europe" => [200, 100, 50, 220],
  "africa" => [220, 180, 50, 220],
  "asia_east" => [200, 50, 100, 220],
  "asia_south" => [150, 50, 150, 220],
  "middle_east" => [180, 100, 50, 220],
  "oceania" => [50, 150, 150, 220],
  "other" => [150, 150, 150, 180]
}

major_hubs = %w[ATL PEK DXB HND LAX ORD LHR CDG DFW DEN JFK FRA AMS IST SIN SYD GRU JNB DEL CAN PVG HKG ICN MEX YYZ MAD BKK CGK KUL ZRH]

map = GD::GIS::Map.new(bbox: :world, zoom: 2, basemap: :carto_light, width: 1000, height: 600)
map.style = GD::GIS::Style.load("light")
font_path = GD::GIS::FontHelper.find("DejaVuSans") || GD::GIS::FontHelper.random
map.style.points[:font] ||= font_path
map.style.points[:size] ||= 10

airports_by_continent = airports.group_by { |a| a[:continent] }

airports_by_continent.each do |continent, list|
  major = list.select { |a| major_hubs.include?(a[:iata]) }
  regional = list - major

  if major.any?
    data = major.map { |a| { lon: a[:lon], lat: a[:lat], label: a[:iata] } }
    layer = GD::GIS::PointsLayer.new(
      data, lon: ->(p) { p[:lon] }, lat: ->(p) { p[:lat] },
      icon: "numeric", label: ->(p) { p[:label] },
      font: font_path, size: 14,
      color: colors[continent] || [180, 100, 100, 220],
      font_color: [255, 255, 255, 255]
    )
    map.instance_variable_get(:@points_layers) << layer
  end

  if regional.any?
    data = regional.map { |a| { lon: a[:lon], lat: a[:lat], label: a[:iata] } }
    layer = GD::GIS::PointsLayer.new(
      data, lon: ->(p) { p[:lon] }, lat: ->(p) { p[:lat] },
      icon: "numeric", label: ->(p) { p[:label] },
      font: font_path, size: 7,
      color: colors[continent] || [150, 150, 150, 150],
      font_color: [255, 255, 255, 255]
    )
    map.instance_variable_get(:@points_layers) << layer
  end
end

map.render

framed = polaroid_frame(map.image, margin: 20, bottom_margin: 90)

framed.text_ft(
  "Global Airports -- Major International Hubs",
  x: 30, y: framed.height - 70,
  font: font_path, size: 14,
  color: GD::Color.rgb(40, 40, 40)
)

framed.text_ft(
  "Color by continent | Larger circles = major hubs | Data: OpenFlights.org",
  x: 30, y: framed.height - 50,
  font: font_path, size: 11,
  color: GD::Color.rgb(80, 80, 80)
)

framed.text_ft(
  "German A. Gimenez Silva <ggerman@gmail.com> | libgd-gis / ruby-libgd",
  x: 30, y: framed.height - 30,
  font: font_path, size: 10,
  color: GD::Color.rgb(120, 120, 120)
)

framed.save("/work/airports_framed.png")
render("airports_framed.png")
puts "Saved: airports_framed.png"

Fetching airport data...
Retrieved 6043 airports


"<img src='airports_framed.png' />"

Saved: airports_framed.png
